# 02 — Popularity Prediction
**Spotify Data Mining | CISC 4631 | Group 3**

### Research Questions
1. Can audio features predict whether a song will be **globally popular**?
2. Can audio features predict whether a song will be popular **within its genre**?
3. Do the same features drive both, or does genre context change what matters?

### Pipeline
1. Load shared data
2. Preprocess & build two label sets
3. Baseline classifiers: KNN, Decision Tree, Naive Bayes
4. Neural Network: PyTorch MLP
5. Compare all models across both targets

> **Prerequisite:** Run `00_data_setup.ipynb` first to generate `data/df_popularity_stratified.csv`.

## 0. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, accuracy_score, f1_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

import warnings
warnings.filterwarnings('ignore')

# Shared constants (keep in sync with 00_data_setup.ipynb)
SEED            = 42
DRIVE_DATA_PATH = '/content/drive/MyDrive/data-mining-spotify-team3/cleanedData'
AUDIO_FEATURES = [
    'danceability', 'energy', 'loudness',
    'speechiness', 'acousticness', 'instrumentalness',
    'liveness', 'valence', 'tempo', 'duration_ms'
]
KEY_FEATURES = ['key', 'mode']
ALL_FEATURES = AUDIO_FEATURES + KEY_FEATURES
LABEL_MAP = {'Low': 0, 'Mid': 1, 'High': 2}

np.random.seed(SEED)
torch.manual_seed(SEED)

## 1. Load Data

In [ ]:
import os
df = pd.read_csv(os.path.join(DRIVE_DATA_PATH, 'df_popularity_stratified.csv'))
print(f'Loaded: {df.shape}')
print('\nGlobal popularity class:')
print(df['popularity_class'].value_counts())
print('\nGenre-relative popularity class:')
print(df['genre_popularity_class'].value_counts())
df.head()

## 2. Preprocess

In [ ]:
# Scale features with StandardScaler
scaler = StandardScaler()
X = pd.DataFrame(
    scaler.fit_transform(df[ALL_FEATURES]),
    columns=ALL_FEATURES
)

# Encode labels
y_global = df['popularity_class'].map(LABEL_MAP)
y_genre  = df['genre_popularity_class'].map(LABEL_MAP)

print('Feature matrix shape:', X.shape)
print('Global label distribution:', y_global.value_counts().to_dict())
print('Genre-relative label distribution:', y_genre.value_counts().to_dict())

In [ ]:
# Stratify split on global label to keep class balance
X_train, X_test, y_train_g, y_test_g, y_train_gr, y_test_gr = train_test_split(
    X, y_global, y_genre,
    test_size=0.2, random_state=SEED, stratify=y_global
)
print(f'Train: {len(X_train):,} | Test: {len(X_test):,}')

## 3. Baseline Classifiers
KNN, Decision Tree, and Naive Bayes — tested on both popularity targets.

In [ ]:
def evaluate(name, model, X_tr, y_tr, X_te, y_te):
    """Fit model, run CV, print classification report, show confusion matrix. Returns model and predictions."""
    model.fit(X_tr, y_tr)
    preds = model.predict(X_te)

    # Cross-validation on training data (class format)
    cv_scores = cross_val_score(model, X_tr, y_tr, cv=10, scoring='accuracy')
    print(f'\n=== {name} ===')
    print("CV Accuracy: %0.2f (+/- %0.2f)" % (cv_scores.mean(), cv_scores.std() * 2))

    print(classification_report(y_te, preds, target_names=['Low', 'Mid', 'High']))
    disp = ConfusionMatrixDisplay(
        confusion_matrix(y_te, preds),
        display_labels=['Low', 'Mid', 'High']
    )
    disp.plot(cmap='Blues')
    plt.title(name)
    plt.show()
    return model, preds

### 3.1 Target: Global Popularity

In [ ]:
print('=' * 50)
print('TARGET: GLOBAL POPULARITY')
print('=' * 50)

knn_g,  preds_knn_g  = evaluate('KNN (k=5) — Global',
    KNeighborsClassifier(n_neighbors=5), X_train, y_train_g, X_test, y_test_g)

dt_g,   preds_dt_g   = evaluate('Decision Tree — Global',
    DecisionTreeClassifier(max_depth=10, random_state=SEED), X_train, y_train_g, X_test, y_test_g)

nb_g,   preds_nb_g   = evaluate('Naive Bayes — Global',
    GaussianNB(), X_train, y_train_g, X_test, y_test_g)

### 3.2 Target: Genre-Relative Popularity

In [ ]:
print('=' * 50)
print('TARGET: GENRE-RELATIVE POPULARITY')
print('=' * 50)

knn_gr, preds_knn_gr = evaluate('KNN (k=5) — Genre-Relative',
    KNeighborsClassifier(n_neighbors=5), X_train, y_train_gr, X_test, y_test_gr)

dt_gr,  preds_dt_gr  = evaluate('Decision Tree — Genre-Relative',
    DecisionTreeClassifier(max_depth=10, random_state=SEED), X_train, y_train_gr, X_test, y_test_gr)

nb_gr,  preds_nb_gr  = evaluate('Naive Bayes — Genre-Relative',
    GaussianNB(), X_train, y_train_gr, X_test, y_test_gr)

## 4. Neural Network (PyTorch MLP)

In [ ]:
class PopularityMLP(nn.Module):
    """
    3-class MLP for popularity prediction.
    Architecture: Linear -> BatchNorm -> ReLU -> Dropout (repeated per hidden layer)
    """
    def __init__(self, input_dim, hidden_dims=None, num_classes=3, dropout=0.3):
        super().__init__()
        if hidden_dims is None:
            hidden_dims = [128, 64, 32]
        layers = []
        in_dim = input_dim
        for h in hidden_dims:
            layers += [
                nn.Linear(in_dim, h),
                nn.BatchNorm1d(h),
                nn.ReLU(),
                nn.Dropout(dropout)
            ]
            in_dim = h
        layers.append(nn.Linear(in_dim, num_classes))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

In [ ]:
def train_mlp(X_tr, y_tr, X_te, y_te, label, epochs=60, lr=1e-3, batch_size=256):
    """
    Train PopularityMLP and report test accuracy every 10 epochs.
    Returns: (model, final_predictions, accuracy_history)
    """
    X_tr_t = torch.tensor(X_tr.values, dtype=torch.float32)
    y_tr_t = torch.tensor(y_tr.values, dtype=torch.long)
    X_te_t = torch.tensor(X_te.values, dtype=torch.float32)
    y_te_t = torch.tensor(y_te.values, dtype=torch.long)

    loader = DataLoader(TensorDataset(X_tr_t, y_tr_t), batch_size=batch_size, shuffle=True)

    model     = PopularityMLP(input_dim=X_tr.shape[1])
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss()
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.5)

    history = []
    for epoch in range(epochs):
        model.train()
        epoch_loss = 0.0
        for xb, yb in loader:
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        scheduler.step()

        if (epoch + 1) % 10 == 0:
            model.eval()
            with torch.no_grad():
                acc = (model(X_te_t).argmax(dim=1) == y_te_t).float().mean().item()
            print(f'[{label}] Epoch {epoch+1:>2}/{epochs} | Loss: {epoch_loss/len(loader):.4f} | Test Acc: {acc:.4f}')
            history.append({'epoch': epoch + 1, 'acc': acc})

    # Final evaluation
    model.eval()
    with torch.no_grad():
        final_preds = model(X_te_t).argmax(dim=1).numpy()

    print(f'\n=== MLP ({label}) ===')
    print(classification_report(y_te, final_preds, target_names=['Low', 'Mid', 'High']))

    disp = ConfusionMatrixDisplay(
        confusion_matrix(y_te, final_preds),
        display_labels=['Low', 'Mid', 'High']
    )
    disp.plot(cmap='Blues')
    plt.title(f'MLP Confusion Matrix \u2014 {label}')
    plt.show()

    return model, final_preds, history

### 4.1 MLP — Global Popularity

In [ ]:
mlp_g, preds_mlp_g, hist_g = train_mlp(
    X_train, y_train_g, X_test, y_test_g, label='Global'
)

### 4.2 MLP — Genre-Relative Popularity

In [ ]:
mlp_gr, preds_mlp_gr, hist_gr = train_mlp(
    X_train, y_train_gr, X_test, y_test_gr, label='Genre-Relative'
)

## 5. Compare All Models

In [ ]:
def scores(preds, y_te):
    return {
        'Accuracy':   round(accuracy_score(y_te, preds), 4),
        'Macro F1':   round(f1_score(y_te, preds, average='macro'), 4),
        'Weighted F1':round(f1_score(y_te, preds, average='weighted'), 4)
    }

# Global target
df_global = pd.DataFrame([
    {'Model': 'KNN',           **scores(preds_knn_g,  y_test_g)},
    {'Model': 'Decision Tree', **scores(preds_dt_g,   y_test_g)},
    {'Model': 'Naive Bayes',   **scores(preds_nb_g,   y_test_g)},
    {'Model': 'MLP',           **scores(preds_mlp_g,  y_test_g)},
])

# Genre-relative target
df_genre_rel = pd.DataFrame([
    {'Model': 'KNN',           **scores(preds_knn_gr,  y_test_gr)},
    {'Model': 'Decision Tree', **scores(preds_dt_gr,   y_test_gr)},
    {'Model': 'Naive Bayes',   **scores(preds_nb_gr,   y_test_gr)},
    {'Model': 'MLP',           **scores(preds_mlp_gr,  y_test_gr)},
])

print('Target: Global Popularity')
print(df_global.to_string(index=False))
print('\nTarget: Genre-Relative Popularity')
print(df_genre_rel.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

for ax, df_res, title in [
    (axes[0], df_global,    'Global Popularity'),
    (axes[1], df_genre_rel, 'Genre-Relative Popularity')
]:
    x = range(len(df_res))
    width = 0.25
    ax.bar([i - width for i in x], df_res['Accuracy']   * 100, width, label='Accuracy')
    ax.bar([i          for i in x], df_res['Macro F1']   * 100, width, label='Macro F1')
    ax.bar([i + width for i in x], df_res['Weighted F1'] * 100, width, label='Weighted F1')
    ax.set_title(title)
    ax.set_xticks(list(x))
    ax.set_xticklabels(df_res['Model'], rotation=15, ha='right')
    ax.set_ylabel('Score (%)')
    ax.set_ylim(0, 100)
    ax.legend()

plt.suptitle('All Models: Global vs Genre-Relative Popularity', fontsize=13)
plt.tight_layout()
plt.show()

### 5.1 MLP Learning Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)

for ax, hist, title in [
    (axes[0], hist_g,  'MLP — Global Popularity'),
    (axes[1], hist_gr, 'MLP — Genre-Relative Popularity')
]:
    epochs = [h['epoch'] for h in hist]
    accs   = [h['acc']   for h in hist]
    ax.plot(epochs, accs, marker='o', color='steelblue')
    ax.set_title(title)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Test Accuracy')
    ax.set_ylim(0, 1)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Summary

**Research Question 1 — Global Popularity**
- Audio features alone have limited predictive power for global popularity (weak linear correlations seen in EDA).
- Non-linear models (Decision Tree, MLP) outperform Naive Bayes, confirming that genre interactions drive signal.

**Research Question 2 — Genre-Relative Popularity**
- Genre-relative labels compare songs within their own genre, removing genre-level baseline differences.
- If accuracy improves on genre-relative labels: audio features carry genre-context signal.
- If accuracy is similar: audio features alone don't capture what makes a song stand out within its genre.

**Research Question 3 — Feature Consistency**
- Compare confusion matrices and per-class F1 across both targets.
- Consistent misclassification patterns (e.g., Low vs Mid harder than Low vs High) appear in both targets,
  suggesting the models capture global audio fingerprints more than genre-internal variation.

**Takeaway:** A song's audio profile is a weak but real predictor of popularity.
Genre context matters — what makes a Classical song stand out is different from what makes a Hip-Hop song stand out.